## 流自定义生成器函数

您可以在 LCEL 管道中使用生成器函数（即使用 yield 关键字且行为类似于迭代器的函数）。

这些生成器的签名应该是 Iterator[Input] -> Iterator[Output] 。或者对于异步生成器： AsyncIterator[Input] -> AsyncIterator[Output] 。

这些对于： - 实现自定义输出解析器 - 修改上一步的输出，同时保留流功能

让我们为逗号分隔列表实现一个自定义输出解析器。

In [1]:
from langchain_openai import ChatOpenAI, OpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
model = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.3,
)

In [2]:
from typing import Iterator, List

from langchain.prompts.chat import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


prompt = ChatPromptTemplate.from_template(
    "响应以CSV的格式返回中文列表，不要返回其他内容。请输出与{transportation}类似的交通工具"
)

In [3]:
str_chain = prompt | model | StrOutputParser()

In [4]:
str_chain.invoke({"transportation":"飞机"})

'<think>\n好的，用户让我用CSV格式返回包含中文列表的“飞机”相关的交通工具，并且不能有任何其他内容。首先，我需要确认用户的需求是什么。他们可能是在做数据处理或者需要整理信息，所以需要结构化的数据。\n\n接下来，我要确保输出符合要求。CSV文件需要有列名和行数据。这里的问题是关于飞机类似的交通工具，但中文列表可能包括其他类型的交通工具，比如火车、汽车等。不过用户特别指出要“飞机”，所以可能需要确认是否有其他交通工具也符合条件，或者是否应该只包含飞机相关的词汇。\n\n然后，我需要检查是否有遗漏的部分。用户明确要求不要返回其他内容，所以必须确保CSV文件中没有额外的文字或格式错误。另外，列名应该是清晰的，比如“交通工具”和“中文名称”，这样用户容易理解数据结构。\n\n还要注意数据的一致性。如果用户希望所有交通工具都包含在内，可能需要将火车、汽车等作为例子。但根据问题描述，用户明确要求的是飞机类似的交通工具，所以可能只需要列出与飞机相关的词汇或例子。例如，可以包括“飞机”、“高铁”、“民航”等。\n\n最后，确保生成的CSV文件正确无误，并且符合格式要求。检查一下是否有拼写错误或者标点符号的问题，然后输出即可。\n</think>\n\n```csv\n交通工具,中文名称\n飞机,飞机\n火车,火车\n汽车,汽车\n高铁,高铁\n民航,民航\n```'

In [5]:
for chunk in str_chain.stream({"transportation":"飞机"}):
    print(chunk, end="", flush=True)

<think>
好的，用户让我用CSV格式返回包含中文列表的飞机相关的交通方式，并且不能有其他内容。首先，我需要确认用户的需求是什么。他们可能是在寻找用于某种数据展示或查询的列表，比如旅游推荐、航班信息或者交通导航。

接下来，我得考虑如何组织这些数据。用户要求的是“中文列表”，所以每个条目必须是中文。同时，要确保输出格式正确，符合CSV标准，即用逗号分隔每一行的数据，并且每行只有一个元素。

然后，我需要确定可能的交通工具类型。飞机通常包括航班、飞机场信息、航线等。但用户明确要求“与飞机类似的交通工具”，所以可能需要列举一些常见的航空相关交通方式，比如航班、机场、航线等，用中文呈现。

需要注意的是，不能包含其他无关的信息，比如日期或具体航班号，因为用户只关心列表本身。同时，要确保数据准确，符合实际的交通术语，避免使用不常用或错误的词汇。

最后，检查一遍输出是否符合CSV格式的要求，没有多余的空格或换行符，并且所有内容都是中文。这样用户就能轻松获取所需的数据了。
</think>

flight,airport,airline,route

In [6]:
# 这是一个自定义解析器，用于拆分 llm 令牌的迭代器
# 放入以逗号分隔的字符串列表中
def split_into_list(input: Iterator[str]) -> Iterator[List[str]]:
    # 保留部分输入，直到得到逗号
    buffer = ""
    for chunk in input:
        # 将当前块添加到缓冲区
        buffer += chunk
        # 当缓冲区中有逗号时
        while "," in buffer:
            # 以逗号分割缓冲区
            comma_index = buffer.index(",")
            # 产生逗号之前的所有内容
            yield [buffer[:comma_index].strip()]
            # 保存其余部分以供下一次迭代使用
            buffer = buffer[comma_index + 1 :]
    # 产生最后一个块
    yield [buffer.strip()]

In [7]:
list_chain = str_chain | split_into_list
for chunk in list_chain.stream({"transportation":"飞机"}):
    print(chunk, end="", flush=True)

['<think>\n好的，用户让我用CSV格式返回包含中文列表的飞机相关的交通方式，而且不能有其他内容。首先，我需要确认用户的需求是什么。他们可能是在寻找一种将中文词汇转换为CSV结构的方式，用于数据处理或者展示。\n\n接下来，我要确保输出符合要求。CSV文件需要列名和行数据。这里的问题是关于交通工具，所以列名应该是“名称”或“类型”。然后，每个行应该包含对应的中文交通方式。\n\n我需要检查是否有重复的词语，避免出现空格或其他格式问题。同时，确认用户是否希望保留所有可能的交通工具，或者只列出常见的。另外，用户特别指出不要返回其他内容，所以必须严格遵循CSV格式，并且不添加任何额外信息。\n\n现在，我需要生成一个符合要求的例子。例如，可以包括“飞机”、“火车”、“汽车”等常见交通工具，每个用中文表示。确保列名正确，并且数据无误。最后，确认输出仅包含CSV内容，没有其他文本。\n</think>\n\nname']['type\n飞机']['飞行器\n火车']['铁路运输\n汽车']['陆地交通']

## 异步版本

In [8]:
from typing import AsyncIterator


async def asplit_into_list(
    input: AsyncIterator[str],
) -> AsyncIterator[List[str]]:  # async def
    buffer = ""
    async for (
        chunk
    ) in input:  # `input` 是一个 `async_generator` 对象，所以使用 `async for`
        buffer += chunk
        while "," in buffer:
            comma_index = buffer.index(",")
            yield [buffer[:comma_index].strip()]
            buffer = buffer[comma_index + 1 :]
    yield [buffer.strip()]


list_chain = str_chain | asplit_into_list

In [9]:
async for chunk in list_chain.astream({"transportation":"飞机"}):
    print(chunk, flush=True)

['<think>\n好的，用户让我用CSV格式返回中文列表，并且不能有其他内容。主题是“飞机类似的交通工具”。首先，我需要确定哪些交通工具可以类比于飞机。常见的类似概念可能包括火车、高铁、轮船或者滑板等。\n\n接下来，我得确保数据的准确性。比如，火车和高铁都是陆地交通，适合在城市或大范围地区使用；轮船则是水路运输，适用于沿海或内陆水域。滑板虽然看起来像飞机，但属于轻便交通工具，可能不太合适。需要确认这些选项是否符合用户的需求。\n\n然后，我需要考虑CSV的格式要求。确保列名正确，数据无误，并且只包含中文内容。同时，用户特别指出不要返回其他内容，所以必须严格遵守这一点。\n\n最后，检查一遍生成的内容是否符合所有条件，没有多余的信息，格式正确。这样用户就能得到他们想要的结果了。\n</think>\n\n飞机类似的交通工具（以CSV格式）：  \n"火车" "高铁" "轮船" "滑板"']


In [10]:
await list_chain.ainvoke({"transportation":"飞机"})

['<think>\n好的，用户让我用CSV格式返回中文列表，而且不能有其他内容。主题是“飞机类似的交通工具”。首先，我需要确定有哪些交通工具可以类比于飞机。通常来说，飞机是四旋翼飞行器，所以可能的选项包括直升机、滑翔机、风筝、滑板车等等。\n\n接下来，我要确保这些交通工具都是中文名称，并且正确无误。比如，直升机在中文里常用“直升机”或“旋翼机”，滑翔机可以叫“滑翔机”。然后，我需要检查每个条目的格式是否正确，包括列名和数据。用户要求输出以CSV的形式，所以必须严格按照CSV的结构来组织。\n\n另外，用户可能希望这些交通工具具有一定的飞行特性，比如悬停、低空飞行等，但不需要详细说明它们的功能，只要列出名称即可。同时，要确保没有多余的信息，只返回结果。最后，确认输出格式正确，没有语法错误或拼写错误。\n</think>\n\nid',
 'transport_type\n1',
 '直升机\n2',
 '滑翔机\n3',
 '风筝\n4',
 '滑板车']